In [1]:
# 프로젝트 루트(= 'data' 폴더가 있는 위치)로 이동. 여러 번 실행해도, 어디서 시작해도 안전.
import os
from pathlib import Path

while not (Path.cwd() / 'data').exists():
    parent = Path.cwd().parent
    if parent == Path.cwd():  # 루트까지 다 올라갔는데도 못 찾은 경우 무한루프 방지
        raise FileNotFoundError("'data' 폴더를 찾지 못했습니다. PRISM 폴더 내부에서 실행 중인지 확인해주세요.")
    os.chdir(parent)

print('현재 작업 디렉토리:', os.getcwd())

현재 작업 디렉토리: c:\Users\rose0\OneDrive\Desktop\2026\대학교\1학기\캡스톤디자인(1)\PRISM


In [2]:
# src 모듈 import 설정
import sys
sys.path.append('..')

import pandas as pd
import colorsys
import plotly.graph_objects as go

from src.scent_map import compute_scent_map, SECTION_ORDER

In [3]:
# 데이터 불러오기 + 좌표/색상 계산
df = pd.read_csv('data/processed/fragella_processed.csv')
df, section_angles = compute_scent_map(df)

print(f'전체 향수 수: {len(df)}개')
for s in SECTION_ORDER:
    v = section_angles[s]
    print(f"  {s:15s} {v['count']:6,}개  {v['start']:6.1f}° ~ {v['end']:6.1f}°")

전체 향수 수: 38488개
  Floral Amber    15,867개     0.0° ~   25.7°
  Soft Amber      30,473개    25.7° ~   61.4°
  Amber           15,747개    61.4° ~   87.0°
  Woody Amber     19,126개    87.0° ~  115.3°
  Woods           26,545개   115.3° ~  148.6°
  Mossy Woods      8,999개   148.6° ~  167.9°
  Dry Woods        8,729개   167.9° ~  187.0°
  Aromatic        21,389개   187.0° ~  216.9°
  Citrus          19,893개   216.9° ~  245.7°
  Water           12,599개   245.7° ~  268.6°
  Green            9,775개   268.6° ~  288.8°
  Fruity          14,385개   288.8° ~  313.3°
  Floral          24,237개   313.3° ~  345.1°
  Soft Floral      5,307개   345.1° ~  360.0°


In [4]:
# HSL -> RGB 변환 (Plotly는 hex/rgb 문자열을 받음)
def hsl_to_rgb_str(h, s, l):
    r, g, b = colorsys.hls_to_rgb(h/360, l/100, s/100)
    return f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'

df['plot_color'] = df.apply(
    lambda row: hsl_to_rgb_str(row['scent_map_hue'], row['scent_map_saturation'], row['scent_map_lightness']),
    axis=1
)

# 마우스오버 시 표시할 텍스트 (이름, 브랜드, 대표 accord)
df['hover_text'] = (
    df['Name'] + '<br>' + df['Brand'].fillna('') +
    '<br>Accords: ' + df['Main Accords'].astype(str)
)

In [5]:
# 배경: 14섹션 경계선 + 이름표 (Plotly shape/annotation)
import math

shapes = []
annotations = []
max_r = 1.05

for s in SECTION_ORDER:
    v = section_angles[s]
    start_rad = math.radians(v['start'])
    x1, y1 = max_r * math.sin(start_rad), max_r * math.cos(start_rad)
    shapes.append(dict(type='line', x0=0, y0=0, x1=x1, y1=y1,
                        line=dict(color='#ddd', width=0.7)))

    mid_rad = math.radians(v['center'])
    lx, ly = 1.15 * math.sin(mid_rad), 1.15 * math.cos(mid_rad)
    from src.scent_map import COLOR_HUE_ANCHOR

    label_r, label_g, label_b = colorsys.hls_to_rgb(COLOR_HUE_ANCHOR[s]/360, 0.42, 0.75)
    label_color = f'rgb({int(label_r*255)},{int(label_g*255)},{int(label_b*255)})'

    annotations.append(dict(x=lx, y=ly, text=f'<b>{s}</b>', showarrow=False,
                            font=dict(size=12, color=label_color)))

In [6]:
import numpy as np

MAX_POINTS_SHOWN = 4000

def sample_for_view(x_range, y_range):
    if x_range is None or y_range is None:
        visible = df
    else:
        visible = df[
            (df['scent_map_x'] >= x_range[0]) & (df['scent_map_x'] <= x_range[1]) &
            (df['scent_map_y'] >= y_range[0]) & (df['scent_map_y'] <= y_range[1])
        ]
    if len(visible) > MAX_POINTS_SHOWN:
        visible = visible.sample(n=MAX_POINTS_SHOWN, random_state=42)
    return visible

def compute_opacity(visible):
    return 1 #(0.15 + 0.6 * visible['scent_map_radius']).clip(upper=0.75)

initial = sample_for_view(None, None)

scatter = go.Scattergl(
    x=initial['scent_map_x'],
    y=initial['scent_map_y'],
    mode='markers',
    marker=dict(
        color=initial['plot_color'],           # df -> initial
        size=3,
        opacity=compute_opacity(initial),       # df -> initial
        line=dict(width=0),
    ),
    text=initial['hover_text'],
    hovertemplate='%{text}<extra></extra>',
)

fig = go.FigureWidget(data=[scatter])
fig.update_layout(
    title='PRISM Scent Map',
    shapes=shapes,
    annotations=annotations,
    xaxis=dict(visible=False, range=[-1.35, 1.35]),
    yaxis=dict(visible=False, range=[-1.35, 1.35], scaleanchor='x'),
    plot_bgcolor='#fafaf8',
    paper_bgcolor='#fafaf8',
    width=800, height=800,
    showlegend=False,
)

def on_zoom(layout, x_range, y_range):
    visible = sample_for_view(x_range, y_range)
    with fig.batch_update():
        fig.data[0].x = visible['scent_map_x']
        fig.data[0].y = visible['scent_map_y']
        fig.data[0].marker.color = visible['plot_color']
        fig.data[0].marker.opacity = compute_opacity(visible)   # 추가된 줄
        fig.data[0].text = visible['hover_text']

fig.layout.on_change(on_zoom, 'xaxis.range', 'yaxis.range')

fig

FigureWidget({
    'data': [{'hovertemplate': '%{text}<extra></extra>',
              'marker': {'color': array(['rgb(202,185,177)', 'rgb(227,225,227)', 'rgb(202,187,174)', ...,
                                         'rgb(221,218,218)', 'rgb(212,210,204)', 'rgb(237,225,33)'], dtype=object),
                         'line': {'width': 0},
                         'opacity': 1,
                         'size': 3},
              'mode': 'markers',
              'text': array(["Icon Man for men<br>Ga-De<br>Accords: ['fresh spicy', 'woody', 'tobacco', 'leather', 'citrus', 'sweet', 'animalic', 'green', 'aromatic', 'musky']",
                             "Christina Aguilera Xtina<br>Christina Aguilera<br>Accords: ['fruity', 'sweet', 'woody', 'gourmand', 'earthy', 'patchouli', 'mossy', 'rose', 'musky', 'amber', 'aquatic']",
                             "Lorenzo Villoresi Firenze Incensi<br>Lorenzo Villoresi<br>Accords: ['amber', 'balsamic', 'warm spicy', 'aromatic', 'green', 'smoky', 'fresh s

In [7]:
# 계산 결과 저장 (Streamlit 앱에서 재사용)
OUTPUT_PATH = 'data/processed/fragella_scent_map.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'저장 완료: {OUTPUT_PATH}')

저장 완료: data/processed/fragella_scent_map.csv


In [8]:
df[df['Name'].str.contains("Humaniste", case=False, na=False)][['Name','scent_map_hue']]

,Name,scent_map_hue
18629,Frapin L'Humaniste,296.35
18641,L'Humaniste Extreme unisex,355.73


In [9]:
import src.scent_map as sm
print(sm.__file__)
print(sm.STRENGTH_WEIGHT)

c:\Users\rose0\OneDrive\Desktop\2026\대학교\1학기\캡스톤디자인(1)\PRISM\src\scent_map.py
{'Dominant': 8, 'Prominent': 4, 'Moderate': 2, 'Subtle': 1}


In [10]:
import time
start = time.time()
df, section_angles = compute_scent_map(df)
print(f'계산 시간: {time.time()-start:.2f}초')

계산 시간: 2.62초


In [11]:
df[df['Name'].isin(['Musc unisex', 'Supercharged Musk unisex'])][['Name','scent_map_hue','scent_map_saturation','scent_map_radius']]

,Name,scent_map_hue,scent_map_saturation,scent_map_radius
26,Supercharged Musk unisex,327.61,91.5,0.9699
3118,Musc unisex,335.39,36.4,0.7008
8501,Musc unisex,329.73,7.4,0.3562
35706,Musc unisex,336.14,90.2,0.9650
38116,Musc unisex,328.55,91.1,0.9684


In [12]:
import ast
from collections import Counter

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return None

freq = Counter()
for notes in df['Notes'].apply(safe_eval):
    if not isinstance(notes, dict):
        continue
    for layer in ('Top', 'Middle', 'Base'):
        for item in notes.get(layer, []):
            if isinstance(item, dict) and item.get('name'):
                freq[item['name'].strip().lower()] += 1

total = sum(freq.values())
sorted_freq = sorted(freq.items(), key=lambda x: -x[1])

print('전체 고유 노트 수:', len(freq))
print('전체 등장 총합:', total)
print()
for n in [50, 100, 200, 300, 400]:
    cum = sum(c for _, c in sorted_freq[:n])
    print(f'상위 {n}개 -> {cum/total*100:.1f}% 커버')

print()
print('상위 400개:')
for name, c in sorted_freq[:400]:
    print(f'  {name}: {c}')

전체 고유 노트 수: 1688
전체 등장 총합: 422039

상위 50개 -> 61.1% 커버
상위 100개 -> 75.6% 커버
상위 200개 -> 87.3% 커버
상위 300개 -> 92.6% 커버
상위 400개 -> 95.5% 커버

상위 400개:
  musk: 19443
  amber: 16542
  sandalwood: 14589
  bergamot: 14552
  jasmine: 12944
  patchouli: 12742
  vanilla: 11799
  rose: 11636
  cedar: 11399
  vetiver: 7148
  mandarin orange: 7094
  lemon: 5778
  tonka bean: 5607
  orange blossom: 4933
  lavender: 4744
  benzoin: 4456
  cardamom: 4313
  geranium: 4303
  iris: 3798
  labdanum: 3674
  violet: 3535
  leather: 3515
  pink pepper: 3511
  grapefruit: 3489
  oakmoss: 3257
  woody notes: 3174
  orange: 2989
  incense: 2899
  cinnamon: 2832
  saffron: 2802
  white musk: 2686
  lily-of-the-valley: 2668
  peach: 2611
  peony: 2589
  freesia: 2580
  tuberose: 2568
  pear: 2404
  heliotrope: 2317
  ylang-ylang: 2299
  nutmeg: 2273
  ginger: 2040
  citruses: 2032
  raspberry: 2020
  pepper: 2017
  neroli: 1992
  mint: 1937
  green notes: 1936
  magnolia: 1855
  black currant: 1794
  apple: 1733
  ou

In [13]:
import ast
from collections import Counter
from src.scent_map import ACCORD_TO_SECTION, AXIS_EXCLUDED_ACCORDS

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return None

# 전체 데이터에서 실제 등장하는 고유 accord와 빈도 집계
raw_df = pd.read_csv('data/processed/fragella_processed.csv')

freq = Counter()
for lst in raw_df['Main Accords'].apply(safe_eval):
    if isinstance(lst, list):
        for a in lst:
            freq[str(a).strip().lower()] += 1

# 현재 코드가 알고 있는 accord (매핑됨 + 제외됨)
mapped = set(ACCORD_TO_SECTION.keys())
excluded = AXIS_EXCLUDED_ACCORDS
accounted = mapped | excluded

all_accords = set(freq.keys())
unaccounted = all_accords - accounted

print('전체 고유 accord 수:', len(all_accords))
print('매핑됨:', len(mapped), '/ 제외됨:', len(excluded))
print('처리 안 된(새로 발견된) accord 수:', len(unaccounted))
print()

if unaccounted:
    print('처리 안 된 accord 목록 (빈도순):')
    for a in sorted(unaccounted, key=lambda x: -freq[x]):
        print(f'  {a}: {freq[a]}건')
else:
    print('새로 발견된 accord 없음 — 기존 매핑이 전체 데이터를 완전히 커버함')

전체 고유 accord 수: 85
매핑됨: 70 / 제외됨: 15
처리 안 된(새로 발견된) accord 수: 0

새로 발견된 accord 없음 — 기존 매핑이 전체 데이터를 완전히 커버함


In [14]:
import ast
from collections import Counter

def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return None

# 원본 CSV를 새로 불러와서 확인 (compute_scent_map 거치기 전 상태)
raw_df = pd.read_csv('data/processed/fragella_processed.csv')

freq = Counter()
total_rows = len(raw_df)
for lst in raw_df['Main Accords'].apply(safe_eval):
    if isinstance(lst, list):
        for a in lst:
            freq[str(a).strip().lower()] += 1

print(f'전체 향수 수: {total_rows}개')
print(f'전체 고유 accord 수: {len(freq)}개')
print()
print(f'{"순위":>4s} {"accord":18s} {"건수":>8s} {"비율":>7s}')
#for i, (a, c) in enumerate(sorted(freq.items(), key=lambda x: -x[1]), 1):
#    print(f'{i:4d} {a:18s} {c:8d} {c/total_rows*100:6.1f}%')
for i, (a, c) in enumerate(sorted(freq.items(), key=lambda x: -x[1]), 1):
    print(f'{a:18s}')

전체 향수 수: 38489개
전체 고유 accord 수: 85개

  순위 accord                   건수      비율
woody             
citrus            
powdery           
sweet             
aromatic          
floral            
warm spicy        
fresh spicy       
amber             
fruity            
musky             
white floral      
gourmand          
vanilla           
fresh             
green             
rose              
earthy            
balsamic          
patchouli         
animalic          
soft spicy        
leather           
oud               
aquatic           
herbal            
lavender          
iris              
violet            
yellow floral     
mossy             
smoky             
tropical          
ozonic            
lactonic          
tuberose          
cinnamon          
marine            
tobacco           
honey             
caramel           
almond            
nutty             
coconut           
aldehydic         
salty             
metallic          
anis              
cacao     